In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
cols_grav = [
    # identifiers
    "iso3_o", "iso3_d", "year",
    
    # distance (lnD) — use harmonic, it's the most standard
    "distw_harmonic",
    
    # # language (Language variable)
    # "comlang_off",    # common official language
    # "comlang_ethno",  # spoken by 9%+ of population in both countries
    
    # optional but useful
    "contig",         # shared border (control variable)
    "comleg_posttrans", # common legal origin — relevant for your Legal variable
    
]

In [4]:
# read the post-2015 version
# Get the select columns from cols
df_post = pd.read_csv("../Raw/Gravity_csv_V202211/Gravity_V202211.csv", nrows=100)
df_post

,year,country_id_o,country_id_d,iso3_o,iso3_d,iso3num_o,iso3num_d,country_exists_o,country_exists_d,gmt_offset_2020_o,...,entry_time_o,entry_time_d,entry_tp_o,entry_tp_d,tradeflow_comtrade_o,tradeflow_comtrade_d,tradeflow_baci,manuf_tradeflow_baci,tradeflow_imf_o,tradeflow_imf_d
0,1948,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1949,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1951,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1952,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1969,ABW,AFG,ABW,AFG,533,4,0,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
96,1970,ABW,AFG,ABW,AFG,533,4,0,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
97,1971,ABW,AFG,ABW,AFG,533,4,0,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98,1972,ABW,AFG,ABW,AFG,533,4,0,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df_post.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 87 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    100 non-null    int64  
 1   country_id_o            100 non-null    object 
 2   country_id_d            100 non-null    object 
 3   iso3_o                  100 non-null    object 
 4   iso3_d                  100 non-null    object 
 5   iso3num_o               100 non-null    int64  
 6   iso3num_d               100 non-null    int64  
 7   country_exists_o        100 non-null    int64  
 8   country_exists_d        100 non-null    int64  
 9   gmt_offset_2020_o       36 non-null     float64
 10  gmt_offset_2020_d       62 non-null     float64
 11  distw_harmonic          36 non-null     float64
 12  distw_arithmetic        36 non-null     float64
 13  distw_harmonic_jh       0 non-null      float64
 14  distw_arithmetic_jh     0 non-null      flo

In [ ]:
cols_lang = ["iso_o", "iso_d", "cle", "csl", "col"]

In [8]:
# read language a well
df_lang = pd.read_stata("../Raw/cepii_ling_web.dta", convert_categoricals=False)
df_lang = df_lang[cols_lang].copy()
df_lang

,iso_o,iso_d,cle,csl,col
0,AFG,ALB,0.086628,0.0000,0
1,AFG,DZA,0.100619,0.0000,0
2,AFG,AND,0.141894,0.0000,0
3,AFG,AGO,0.140295,0.0000,0
4,AFG,AIA,0.109629,0.0000,0
...,...,...,...,...,...
37825,ZWE,VUT,0.129664,0.3528,1
37826,ZWE,VEN,0.051485,0.0000,0
37827,ZWE,VNM,0.025630,0.0000,0
37828,ZWE,YEM,0.095582,0.0000,0


In [ ]:
# Rename columns to match iso column names
df_lang = df_lang.rename(columns={"iso_o": "iso3_o", "iso_d": "iso3_d"})

In [10]:
# Check intersection of iso3_o and iso3_d in both dataframes
combinations = set(zip(df_post["iso3_o"], df_post["iso3_d"])).intersection(set(zip(df_lang["iso3_o"], df_lang["iso3_d"])))
print(f"Number of matching combinations: {len(combinations)} out of {len(set(zip(df_post['iso3_o'], df_post['iso3_d'])))} in df_post and {len(set(zip(df_lang['iso3_o'], df_lang['iso3_d'])))} in df_lang")

Number of matching combinations: 1 out of 2 in df_post and 37830 in df_lang


In [11]:
# Join them together
df_gravity = df_post.merge(
    df_lang,
    left_on=["iso3_o", "iso3_d"],
    right_on=["iso3_o", "iso3_d"],
    how="left"
)
df_gravity

,year,country_id_o,country_id_d,iso3_o,iso3_d,iso3num_o,iso3num_d,country_exists_o,country_exists_d,gmt_offset_2020_o,...,entry_tp_d,tradeflow_comtrade_o,tradeflow_comtrade_d,tradeflow_baci,manuf_tradeflow_baci,tradeflow_imf_o,tradeflow_imf_d,cle,csl,col
0,1948,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1949,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1951,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1952,ABW,ABW,ABW,ABW,533,533,0,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1969,ABW,AFG,ABW,AFG,533,4,0,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.128432,0.0,0.0
96,1970,ABW,AFG,ABW,AFG,533,4,0,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.128432,0.0,0.0
97,1971,ABW,AFG,ABW,AFG,533,4,0,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.128432,0.0,0.0
98,1972,ABW,AFG,ABW,AFG,533,4,0,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.128432,0.0,0.0


In [12]:
df_gravity.describe().round(2)

,year,iso3num_o,iso3num_d,country_exists_o,country_exists_d,gmt_offset_2020_o,gmt_offset_2020_d,distw_harmonic,distw_arithmetic,distw_harmonic_jh,...,entry_tp_d,tradeflow_comtrade_o,tradeflow_comtrade_d,tradeflow_baci,manuf_tradeflow_baci,tradeflow_imf_o,tradeflow_imf_d,cle,csl,col
count,100.00,100.0,100.00,100.00,100.00,36.0,62.00,36.0,36.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,26.00,26.0,26.0
mean,1978.26,533.0,395.46,0.36,0.62,-4.0,-0.44,32.0,32.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.13,0.0,0.0
std,21.63,0.0,233.21,0.48,0.49,0.0,4.23,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00,0.0,0.0
min,1948.00,533.0,4.00,0.00,0.00,-4.0,-4.00,32.0,32.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.13,0.0,0.0
25%,1960.00,533.0,4.00,0.00,0.00,-4.0,-4.00,32.0,32.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.13,0.0,0.0
50%,1972.50,533.0,533.00,0.00,1.00,-4.0,-4.00,32.0,32.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.13,0.0,0.0
75%,1996.25,533.0,533.00,1.00,1.00,-4.0,4.50,32.0,32.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.13,0.0,0.0
max,2021.00,533.0,533.00,1.00,1.00,-4.0,4.50,32.0,32.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.13,0.0,0.0


In [13]:
df_gravity = df_gravity.rename(columns={
    "distw_harmonic":   "distance_km",
    "contig":           "shared_border",
    "comleg_posttrans": "common_legal_origin",
    "cle":              "language_proximity",
    "csl":              "language_spoken_share",
    "col":              "common_official_language",
})

In [14]:
# Save to clean data
path = "/Users/jesper/Desktop/CBS/Thesis 1/Jesper-Liedholm-Thesis-Code/Data/Clean/cepii_gravity_language.csv"
df_gravity.to_csv(path, index=False)